# 1. Read source data

In [35]:
import sys
import pandas as pd
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)



In [36]:
from mhcgnomes import parse
from config import RESULT_COLUMNS

DATA_PATH = "D:/Google Drive/Work/ITMO/data/schreiber_full/"
PREDICTION_PATH = "D:/Google Drive/Work/ITMO/MHC-Epitop-Prediction/prediction_results/schreiber_full/"
AUX_PREDICTION_PATH = "D:/Google Drive/Work/ITMO/MHC-Epitop-Prediction/prediction_results/schreiber_12_mers/"


source_total_df =  pd.read_csv("{}schreiber_total_data_full.csv".format(DATA_PATH),sep=";")
ALLELES = list(set(source_total_df['allele']))
per_allele_mt_df =dict()
per_allele_wt_df = dict()
for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    df_mt = pd.read_csv("{}schreiber_data_full_{}_mt.csv".format(DATA_PATH,target_allele_name),sep=";")
    df_wt = pd.read_csv("{}schreiber_data_full_{}_wt.csv".format(DATA_PATH,target_allele_name),sep=";")

    per_allele_mt_df[allele_name] = df_mt
    per_allele_wt_df[allele_name] = df_wt

    assert 'peptide' in df_mt.columns
    assert 'length' in df_mt.columns
    assert RESULT_COLUMNS['id'] in df_mt.columns
    assert 'peptide' in df_wt.columns
    assert 'length' in df_wt.columns
    assert RESULT_COLUMNS['id'] in df_wt.columns


def defineOrganism(allele_name):
    parse_result = parse(allele_name)
    if parse_result.is_mouse:
        return "Mouse"
    elif parse_result.is_human:
        return "Human"
    else:
        print("organism not found")
        return None


def defineClass(allele_name):
    parse_result = parse(allele_name)
    if parse_result.has_mhc_class:
        if parse_result.is_class1:
            return 'I'
        elif parse_result.is_class2:
            return 'II'
        else:
            print("Not class I or class II???!")
    else:
        print("Result without class")
    return None
ALLELES

['H-2-IEd', 'H-2-IAd', 'H-2-Ld', 'H-2-Kd', 'H-2-Dd']

# 3. Read Joined predictions

In [37]:
import os.path

per_allele_predictions = dict()

for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    total_df = pd.read_csv("{}total_predictions_allele_{}.csv".format(PREDICTION_PATH,target_allele_name),sep=";")
    per_allele_predictions[allele_name] = total_df


for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    aux_file_name = "{}total_predictions_allele_{}.csv".format(AUX_PREDICTION_PATH,target_allele_name)
    if os.path.isfile(aux_file_name):
        aux_df = pd.read_csv(aux_file_name,sep=";")
        total_df =  per_allele_predictions[allele_name]
        per_allele_predictions[allele_name] = total_df.append(aux_df)





class_I_total_df = pd.read_csv("{}total_predictions_class_I.csv".format(PREDICTION_PATH),sep=";" )
class_II_total_df  = pd.read_csv("{}total_predictions_class_II.csv".format(PREDICTION_PATH),sep=";")

aux_file_name = "{}total_predictions_class_I.csv".format(AUX_PREDICTION_PATH)
if os.path.isfile(aux_file_name):
    class_I_aux_total_df =  pd.read_csv(aux_file_name,sep=";" )
    class_I_total_df = class_I_total_df.append(class_I_aux_total_df)
aux_file_name = "{}total_predictions_class_II.csv".format(AUX_PREDICTION_PATH)
if os.path.isfile(aux_file_name):
    class_II_aux_total_df =  pd.read_csv(aux_file_name,sep=";" )
    class_II_total_df = class_I_total_df.append(class_II_aux_total_df)

In [38]:
class_I_total_df.tail()

,mutant_id,allele,length,median_percentile,mean_percentile,median_percentile_delta,mean_percentile_delta,median_score_ratio,A20_RPKM_mut,peptide_mt,...,mhcflurry_affinity_wt,mhcflurry_processing_score_wt,mhcflurry_score_el_wt,mhcnuggets_score_wt,mhcnuggets_affinity_wt,puffin_score_wt,puffin_epistemic_var_wt,puffin_aleatoric_var_wt,puffin_binding_likelihood_wt,source_sequence_name.1
8194,Rbbp7-P133L,H-2-Kd,12,22.274875,32.384363,-7.542299,-4.481899,0.736199,52.215983,ARYMLQNPHIIA,...,25964.231219,0.548389,0.026362,0.043346,31281.47,0.123273,0.000564,0.021754,0.051124,seq2728
8195,Rbbp7-P133L,H-2-Kd,12,5.041709,5.205855,-1.812041,-1.541270,0.635363,52.215983,RYMLQNPHIIAT,...,9847.543613,0.733759,0.117999,0.247849,3422.44,0.227844,0.002134,0.044999,0.183067,seq2729
8196,Rbbp7-P133L,H-2-Kd,12,12.700921,11.693961,2.061750,7.822740,1.090890,52.215983,YMLQNPHIIATK,...,23630.547348,0.624764,0.037420,0.305889,1826.45,0.108734,0.000509,0.018197,0.035181,seq2730
8197,Rbbp7-P133L,H-2-Kd,12,62.079125,66.550776,-19.934250,-27.592529,0.490383,52.215983,MLQNPHIIATKT,...,30386.307480,0.373676,0.012370,0.160688,8788.23,0.098523,0.000624,0.015867,0.026575,seq2731
8198,Rbbp7-P133L,H-2-Kd,12,37.660524,45.123762,15.587000,9.095531,1.331511,52.215983,LQNPHIIATKTP,...,30798.239767,0.064657,0.004125,0.051289,28705.56,0.106812,0.000393,0.018136,0.036965,seq2732


## Group by mutant_id and take max scoring

In [39]:
per_allele_grouped_by_predictions = dict()
for allele_name in ALLELES:
    total_df =  per_allele_predictions[allele_name]
    per_allele_grouped_by_predictions[allele_name] = total_df.sort_values('mean_percentile').drop_duplicates(['mutant_id'])


class_I_total_df_grouped_by = class_I_total_df.sort_values('mean_percentile').drop_duplicates(['mutant_id'])
class_II_total_df_grouped_by = class_II_total_df.sort_values('mean_percentile').drop_duplicates(['mutant_id'])


In [40]:
class_I_total_df.loc[class_I_total_df.groupby("mean_percentile")["mean_percentile"].idxmin()]

,mutant_id,allele,length,median_percentile,mean_percentile,median_percentile_delta,mean_percentile_delta,median_score_ratio,A20_RPKM_mut,peptide_mt,...,mhcflurry_affinity_wt,mhcflurry_processing_score_wt,mhcflurry_score_el_wt,mhcnuggets_score_wt,mhcnuggets_affinity_wt,puffin_score_wt,puffin_epistemic_var_wt,puffin_aleatoric_var_wt,puffin_binding_likelihood_wt,source_sequence_name.1
17730,Rbbp7-P133L,H-2-Kd,9,0.051489,0.068182,-0.036674,-0.050087,0.943138,52.215983,RYMLQNPHI,...,37.609165,0.791169,0.973399,0.664461,37.73,0.710995,0.005976,0.058352,8.603114e-01,seq465
31,E2f8-G517A,H-2-Dd,9,0.051125,0.069856,-0.003000,0.041791,0.760641,4.277044,SAPSYAIYL,...,191.447858,0.466283,0.704662,0.703579,24.71,0.734383,0.009045,0.048440,8.867047e-01,seq2032
31,Scly-A424P,H-2-Dd,12,63.477250,54.528014,10.729918,7.303959,1.345439,1.308955,DLKQPVAQLEGR,...,27123.830995,0.246120,0.008827,0.026974,37343.95,0.017289,0.000076,0.001029,3.534648e-09,seq31
17476,Zfp984-K196N,H-2-Kd,9,0.054375,0.070262,0.022584,0.056917,1.039217,5.234282,KYLTQRENL,...,55.655188,0.732204,0.953047,0.493631,239.56,0.544344,0.008406,0.074617,6.546292e-01,seq0
9024,Gba2-N96S,H-2-Ld,10,0.067375,0.101337,-0.009000,-0.027031,0.983684,0.450897,RPFQANSISL,...,225.028273,0.906661,0.906424,0.742429,16.23,0.562253,0.018143,0.060006,7.003423e-01,seq663
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5756,Esco1-D584A,H-2-Dd,8,19.999250,36.994776,0.553500,-4.891982,1.121547,2.033827,QLKHASHL,...,27715.895869,0.032960,0.004086,0.052707,28268.48,0.048894,0.000710,0.004815,6.359514e-04,seq23232
5756,Eif3j1-Y253H,H-2-Kd,12,98.809802,98.609276,-3.196500,-10.185232,0.978631,10.932544,DLADYGGYEGGH,...,27058.058093,0.032425,0.004174,0.011325,44233.94,0.042671,0.000651,0.003898,1.058295e-04,seq290
19738,Flnb-V795G,H-2-Kd,8,98.756750,98.783276,-39.075465,-32.736607,0.734247,10.735158,LSDDEEDG,...,24690.666126,0.003379,0.004117,0.047216,29998.76,0.063448,0.000517,0.007971,2.692490e-03,seq3635
8073,Lrch3-P18L,H-2-Dd,10,45.972337,46.477668,7.108500,5.864000,1.258241,2.321927,VAAAEYSGLV,...,20139.736590,0.002692,0.005001,0.027907,36968.89,0.038288,0.000244,0.003450,7.298822e-05,seq25549


## Save total pred to file

In [41]:
# for decoy_key in DECOY_PREFIXES:
for allele_name in ALLELES:
    total_df = per_allele_predictions[allele_name]
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    total_df.to_csv("{}total_predictions_allele_{}.csv".format(PREDICTION_PATH,target_allele_name),sep=";", index=False)

In [42]:
if class_I_total_df is not None:
    class_I_total_df.to_csv("{}total_predictions_class_I.csv".format(PREDICTION_PATH),sep=";", index=False)
if class_II_total_df is not None:
    class_II_total_df.to_csv("{}total_predictions_class_II.csv".format(PREDICTION_PATH),sep=";", index=False)

In [43]:
# for decoy_key in DECOY_PREFIXES:
for allele_name in ALLELES:
    total_df = per_allele_grouped_by_predictions[allele_name]
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    total_df.to_csv("{}total_predictions_top_for_mutant_id_allele_{}.csv".format(PREDICTION_PATH,target_allele_name),sep=";", index=False)

In [44]:
if class_I_total_df is not None:
    class_I_total_df_grouped_by.to_csv("{}total_predictions_top_for_mutant_id_class_I.csv".format(PREDICTION_PATH),sep=";", index=False)
if class_II_total_df is not None:
    class_II_total_df_grouped_by.to_csv("{}total_predictions_top_for_mutant_id_class_II.csv".format(PREDICTION_PATH),sep=";", index=False)